# Neural Network Fundamentals

This notebook explores the core concepts of neural networks including neurons and activation functions. We'll cover both theoretical aspects and practical implementations to develop a solid foundation in understanding how neural networks work.

## 1. Import Required Libraries

In [ ]:
# Import core libraries for computation and visualization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

# For interactive visualizations
from ipywidgets import interact, interactive, fixed, widgets

# Attempt to import deep learning libraries if available
try:
    import tensorflow as tf
    tf_available = True
    print("TensorFlow is available (version", tf.__version__, ")")
except ImportError:
    tf_available = False
    print("TensorFlow is not available. Some examples will use NumPy only.")

try:
    import torch
    torch_available = True
    print("PyTorch is available (version", torch.__version__, ")")
except ImportError:
    torch_available = False
    print("PyTorch is not available. Some examples will use NumPy only.")

# Set plotting style
sns.set(style="whitegrid")
plt.style.use('seaborn-v0_8')

# For consistent reproducible results
np.random.seed(42)

## 2. Neural Network Architecture

Neural networks are composed of interconnected layers of computational units called neurons. A basic neural network consists of:

1. **Input Layer**: Receives the raw input data
2. **Hidden Layer(s)**: Performs computations and feature extraction
3. **Output Layer**: Produces the final prediction or classification

Let's visualize a simple neural network architecture:

In [ ]:
def plot_neural_network(num_inputs=3, num_hidden=4, num_outputs=2):
    """
    Plots a simple neural network with given number of neurons in each layer
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.gca()
    ax.axis('off')
    
    # Coordinates
    layer_spacing = 4
    neuron_spacing = 2
    
    # Create positions for each neuron
    input_positions = [(1, i*neuron_spacing) for i in range(num_inputs)]
    hidden_positions = [(1+layer_spacing, i*neuron_spacing + (num_inputs-num_hidden)*neuron_spacing/2) 
                        for i in range(num_hidden)]
    output_positions = [(1+2*layer_spacing, i*neuron_spacing + (num_inputs-num_outputs)*neuron_spacing/2) 
                         for i in range(num_outputs)]
    
    # Draw neurons
    for positions, color, name in [(input_positions, 'skyblue', 'Input Layer'), 
                                   (hidden_positions, 'lightgreen', 'Hidden Layer'), 
                                   (output_positions, 'salmon', 'Output Layer')]:
        # Draw circles for neurons
        for x, y in positions:
            circle = plt.Circle((x, y), 0.5, fill=True, color=color, alpha=0.7)
            ax.add_patch(circle)
            
        # Add label for the layer
        layer_x = positions[0][0]
        layer_y = positions[-1][1] + 1.5
        plt.text(layer_x, layer_y, name, ha='center', fontsize=14)
    
    # Draw connections between layers
    for start_layer, end_layer in [(input_positions, hidden_positions), (hidden_positions, output_positions)]:
        for start_x, start_y in start_layer:
            for end_x, end_y in end_layer:
                ax.plot([start_x + 0.5, end_x - 0.5], [start_y, end_y], 'gray', alpha=0.5)
    
    plt.xlim(0, 2 + 2*layer_spacing)
    plt.ylim(-1, max(num_inputs, num_hidden, num_outputs)*neuron_spacing + 1)
    plt.title('Simple Neural Network Architecture', fontsize=16)
    plt.show()

# Plot a simple neural network with 3 inputs, 4 hidden neurons, and 2 outputs
plot_neural_network()

## 3. Neurons: The Building Blocks

A neuron (or perceptron in its simplest form) is the fundamental unit of a neural network. It receives multiple input signals, processes them, and produces an output signal.

### Structure of a Neuron:

1. **Inputs ($x_1, x_2, ..., x_n$)**: Data features or outputs from previous layer neurons
2. **Weights ($w_1, w_2, ..., w_n$)**: Parameters that determine the importance of each input
3. **Bias ($b$)**: Constant term that allows the neuron to fit the data better
4. **Weighted Sum**: $z = w_1x_1 + w_2x_2 + ... + w_nx_n + b = \mathbf{w}^T\mathbf{x} + b$
5. **Activation Function ($\sigma$)**: Non-linear function that transforms the weighted sum
6. **Output**: $y = \sigma(z)$, the result of applying the activation function

Let's visualize a single neuron and implement its functionality:

In [ ]:
def plot_neuron():
    """
    Visualize a single neuron with inputs, weights, bias and activation
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    
    # Draw input signals
    input_points = [(1, i) for i in range(1, 4)]
    for i, (x, y) in enumerate(input_points):
        circle = plt.Circle((x, y), 0.3, fill=True, color='skyblue', alpha=0.7)
        ax.add_patch(circle)
        plt.text(x, y, f"$x_{i+1}$", ha='center', va='center')
        
    # Draw the neuron
    neuron_pos = (4, 2)
    circle = plt.Circle(neuron_pos, 0.5, fill=True, color='lightgreen', alpha=0.7)
    ax.add_patch(circle)
    
    # Draw connections with weights
    for i, (x, y) in enumerate(input_points):
        ax.arrow(x + 0.3, y, 3 - 0.3, 2 - y, head_width=0.1, head_length=0.2, 
                fc='gray', ec='gray', alpha=0.7)
        midx = (x + 0.3 + 3) / 2 + 0.3
        midy = (y + 2) / 2 + 0.3
        plt.text(midx, midy, f"$w_{i+1}$", fontsize=12)
    
    # Draw bias
    ax.arrow(4, 0.5, 0, 1, head_width=0.1, head_length=0.2, fc='gray', ec='gray', alpha=0.7)
    plt.text(4.2, 1, "bias $(b)$", fontsize=12)
    
    # Drawing weighted sum and activation
    plt.text(4, 2, "$\\sum$", ha='center', va='center')
    
    # Draw activation function
    ax.arrow(4.5, 2, 1, 0, head_width=0.1, head_length=0.2, fc='gray', ec='gray', alpha=0.7)
    plt.text(5, 2.3, "$\\sigma()$", ha='center')
    
    # Draw output
    ax.arrow(5.5, 2, 1, 0, head_width=0.1, head_length=0.2, fc='gray', ec='gray', alpha=0.7)
    circle = plt.Circle((7, 2), 0.3, fill=True, color='salmon', alpha=0.7)
    ax.add_patch(circle)
    plt.text(7, 2, "$y$", ha='center', va='center')
    
    # Add equations at the bottom
    plt.text(4, 4.5, "Weighted Sum: $z = w_1x_1 + w_2x_2 + w_3x_3 + b$", fontsize=14, ha='center')
    plt.text(4, 4, "Output: $y = \\sigma(z)$", fontsize=14, ha='center')
    
    plt.xlim(0, 8)
    plt.ylim(0, 5)
    plt.title('Structure of a Single Neuron', fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize a neuron
plot_neuron()

In [ ]:
class Neuron:
    """
    Simple implementation of a single neuron
    """
    def __init__(self, n_inputs):
        """Initialize a neuron with random weights and bias"""
        # Initialize weights with small random values
        self.weights = np.random.randn(n_inputs) * 0.1
        # Initialize bias to zero
        self.bias = 0
        
    def weighted_sum(self, inputs):
        """Calculate the weighted sum of inputs"""
        return np.dot(inputs, self.weights) + self.bias
    
    def activate(self, inputs, activation_fn=None):
        """Apply activation function to weighted sum"""
        z = self.weighted_sum(inputs)
        
        # Default to sigmoid activation if none is provided
        if activation_fn is None:
            return 1 / (1 + np.exp(-z))  # sigmoid function
        else:
            return activation_fn(z)

# Create a neuron with 3 inputs
neuron = Neuron(3)
print("Neuron weights:", neuron.weights)
print("Neuron bias:", neuron.bias)

# Test the neuron with sample inputs
inputs = np.array([0.5, 0.3, 0.2])
output = neuron.activate(inputs)
print(f"\nFor inputs {inputs}:")
print(f"Weighted sum: {neuron.weighted_sum(inputs):.4f}")
print(f"Output after sigmoid activation: {output:.4f}")

## 4. Activation Functions

Activation functions introduce non-linearity into neural networks, allowing them to learn complex patterns. Without activation functions, neural networks would just be linear regression models regardless of depth.

Let's examine the most common activation functions:

1. **Sigmoid**: $\sigma(x) = \frac{1}{1 + e^{-x}}$
2. **Tanh (Hyperbolic Tangent)**: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
3. **ReLU (Rectified Linear Unit)**: $ReLU(x) = \max(0, x)$
4. **Leaky ReLU**: $LeakyReLU(x) = \max(\alpha x, x)$ where $\alpha$ is small (e.g., 0.01)
5. **Softmax**: $softmax(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$

Let's implement these functions and visualize them:

In [ ]:
# Implement activation functions
def sigmoid(x):
    """Sigmoid activation function"""
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    """Derivative of sigmoid function"""
    s = sigmoid(x)
    return s * (1 - s)

def tanh_activation(x):
    """Tanh activation function"""
    return np.tanh(x)

def tanh_derivative(x):
    """Derivative of tanh function"""
    return 1 - np.tanh(x)**2

def relu(x):
    """ReLU activation function"""
    return np.maximum(0, x)

def relu_derivative(x):
    """Derivative of ReLU function"""
    return np.where(x > 0, 1, 0)

def leaky_relu(x, alpha=0.01):
    """Leaky ReLU activation function"""
    return np.maximum(alpha * x, x)

def leaky_relu_derivative(x, alpha=0.01):
    """Derivative of Leaky ReLU function"""
    return np.where(x > 0, 1, alpha)

def softmax(x):
    """Softmax activation function for multi-class classification"""
    # For numerical stability, subtract max value
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

# Create a range of input values
x = np.linspace(-5, 5, 1000)

# Create a figure for comparing activation functions
plt.figure(figsize=(15, 10))

# Plot each activation function
plt.subplot(2, 3, 1)
plt.plot(x, sigmoid(x))
plt.grid(True)
plt.title('Sigmoid')
plt.xlabel('Input')
plt.ylabel('Output')

plt.subplot(2, 3, 2)
plt.plot(x, tanh_activation(x))
plt.grid(True)
plt.title('Tanh')
plt.xlabel('Input')
plt.ylabel('Output')

plt.subplot(2, 3, 3)
plt.plot(x, relu(x))
plt.grid(True)
plt.title('ReLU')
plt.xlabel('Input')
plt.ylabel('Output')

plt.subplot(2, 3, 4)
plt.plot(x, leaky_relu(x))
plt.grid(True)
plt.title('Leaky ReLU (alpha=0.01)')
plt.xlabel('Input')
plt.ylabel('Output')

# Compare sigmoid and tanh together
plt.subplot(2, 3, 5)
plt.plot(x, sigmoid(x), label='Sigmoid')
plt.plot(x, tanh_activation(x), label='Tanh')
plt.grid(True)
plt.title('Sigmoid vs Tanh')
plt.xlabel('Input')
plt.ylabel('Output')
plt.legend()

# For softmax, let's use a different approach since it works on vectors
# We'll show softmax with different temperature values
def softmax_with_temp(x, temp=1.0):
    e_x = np.exp(x / temp)
    return e_x / e_x.sum()

# Create a simple vector
x_softmax = np.array([0.3, 2.9, 1.0, 0.4, 1.7])
temps = [0.2, 0.5, 1.0, 2.0, 5.0]

plt.subplot(2, 3, 6)
for temp in temps:
    sm = softmax_with_temp(x_softmax, temp)
    plt.bar(np.arange(len(x_softmax)) + (temp/10), sm, width=0.1, alpha=0.7, label=f'T={temp}')

plt.title('Softmax with Different Temperatures')
plt.xlabel('Class Index')
plt.ylabel('Probability')
plt.xticks(np.arange(len(x_softmax)))
plt.legend()

plt.tight_layout()
plt.show()

### Derivatives of Activation Functions

The derivatives of activation functions play a crucial role in neural network training through backpropagation. Let's visualize these derivatives:

In [ ]:
# Create a figure for visualizing derivatives
plt.figure(figsize=(15, 10))

# Plot each activation function with its derivative
plt.subplot(2, 2, 1)
plt.plot(x, sigmoid(x), label='Sigmoid')
plt.plot(x, sigmoid_derivative(x), label='Derivative')
plt.grid(True)
plt.title('Sigmoid and its Derivative')
plt.xlabel('Input')
plt.ylabel('Output')
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(x, tanh_activation(x), label='Tanh')
plt.plot(x, tanh_derivative(x), label='Derivative')
plt.grid(True)
plt.title('Tanh and its Derivative')
plt.xlabel('Input')
plt.ylabel('Output')
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(x, relu(x), label='ReLU')
plt.plot(x, relu_derivative(x), label='Derivative')
plt.grid(True)
plt.title('ReLU and its Derivative')
plt.xlabel('Input')
plt.ylabel('Output')
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(x, leaky_relu(x), label='Leaky ReLU')
plt.plot(x, leaky_relu_derivative(x), label='Derivative')
plt.grid(True)
plt.title('Leaky ReLU and its Derivative')
plt.xlabel('Input')
plt.ylabel('Output')
plt.legend()

plt.tight_layout()
plt.show()

### Activation Function Characteristics and Use Cases

| Activation Function | Range | Characteristics | Common Uses | Pros | Cons |
|---------------------|-------|-----------------|-------------|------|------|
| **Sigmoid** | (0, 1) | Smooth, S-shaped | Binary classification, output layer | - Smooth gradient<br>- Output bounded | - Vanishing gradient<br>- Not zero-centered |
| **Tanh** | (-1, 1) | S-shaped, zero-centered | Hidden layers, recurrent networks | - Zero-centered<br>- Stronger gradients than sigmoid | - Still has vanishing gradient issue |
| **ReLU** | [0, ∞) | Linear for positive values, zero for negative | Hidden layers in CNNs and deep networks | - No vanishing gradient for positive values<br>- Computationally efficient | - "Dying ReLU" problem<br>- Not zero-centered |
| **Leaky ReLU** | (-∞, ∞) | Similar to ReLU but with small slope for negative values | Hidden layers | - Prevents dying ReLU<br>- All benefits of ReLU | - May not always outperform ReLU |
| **Softmax** | (0, 1) with sum=1 | Converts values to probability distribution | Multi-class classification output layer | - Good for probability interpretation<br>- Differentiable | - Computationally expensive |

## 5. Implementing a Simple Neuron

Now let's implement a simple neuron and see how it responds to different inputs with various activation functions. We'll create an interactive visualization to see how changing weights affects the neuron's output:

In [ ]:
def visualize_neuron_response(w1=0.5, w2=0.5, bias=0.0, activation='sigmoid'):
    """
    Interactive visualization of neuron response to different inputs
    with adjustable weights and bias
    """
    # Define activation functions dictionary
    activations = {
        'sigmoid': sigmoid,
        'tanh': tanh_activation,
        'relu': relu,
        'leaky_relu': leaky_relu
    }
    
    # Create a grid of input values
    x1 = np.linspace(-2, 2, 100)
    x2 = np.linspace(-2, 2, 100)
    X1, X2 = np.meshgrid(x1, x2)
    
    # Calculate neuron's output for each input pair
    Z = np.zeros_like(X1)
    for i in range(len(x1)):
        for j in range(len(x2)):
            # Weighted sum
            z = X1[i, j] * w1 + X2[i, j] * w2 + bias
            # Activation
            Z[i, j] = activations[activation](z)
    
    # Create 3D plot
    fig = plt.figure(figsize=(12, 10))
    
    # 3D surface plot
    ax1 = fig.add_subplot(221, projection='3d')
    surf = ax1.plot_surface(X1, X2, Z, cmap=cm.viridis, alpha=0.8)
    ax1.set_xlabel('Input 1')
    ax1.set_ylabel('Input 2')
    ax1.set_zlabel('Output')
    ax1.set_title(f'Neuron Output with {activation.capitalize()} Activation')
    
    # Add color bar
    fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=5)
    
    # Contour plot (top-down view)
    ax2 = fig.add_subplot(222)
    contour = ax2.contourf(X1, X2, Z, 20, cmap='viridis')
    ax2.set_xlabel('Input 1')
    ax2.set_ylabel('Input 2')
    ax2.set_title('Contour View (Decision Boundary)')
    fig.colorbar(contour, ax=ax2)
    
    # Plot along x1 axis (for fixed x2=0)
    ax3 = fig.add_subplot(223)
    ax3.plot(x1, [activations[activation](w1*x + w2*0 + bias) for x in x1])
    ax3.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax3.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    ax3.set_xlabel('Input 1 (with Input 2 = 0)')
    ax3.set_ylabel('Output')
    ax3.set_title('Response Along Input 1 Axis')
    ax3.grid(True)
    
    # Plot along x2 axis (for fixed x1=0)
    ax4 = fig.add_subplot(224)
    ax4.plot(x2, [activations[activation](w1*0 + w2*x + bias) for x in x2])
    ax4.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax4.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    ax4.set_xlabel('Input 2 (with Input 1 = 0)')
    ax4.set_ylabel('Output')
    ax4.set_title('Response Along Input 2 Axis')
    ax4.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Display the equation
    equation = f"Output = {activation}({w1:.2f} × Input_1 + {w2:.2f} × Input_2 + {bias:.2f})"
    print(f"Neuron equation: {equation}")

# Create interactive widgets
interact(visualize_neuron_response, 
         w1=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.5, description='Weight 1:'),
         w2=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.5, description='Weight 2:'),
         bias=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.0, description='Bias:'),
         activation=widgets.Dropdown(options=['sigmoid', 'tanh', 'relu', 'leaky_relu'], 
                                    value='sigmoid', description='Activation:'));

## 6. Forward Propagation

Forward propagation is the process of passing input data through the neural network to obtain predictions. For a multi-layer network, the output of one layer becomes the input to the next.

### Mathematical Representation:

For a neural network with L layers:
- Layer 1 (input layer): $a^{[0]} = X$ (input data)
- Layer l (1 ≤ l ≤ L):
  - $z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}$
  - $a^{[l]} = g^{[l]}(z^{[l]})$
  
Where:
- $a^{[l]}$ is the activation output of layer l
- $W^{[l]}$ is the weight matrix of layer l
- $b^{[l]}$ is the bias vector of layer l
- $g^{[l]}$ is the activation function of layer l

Let's implement forward propagation for a simple 2-layer neural network:

In [ ]:
class SimpleNeuralNetwork:
    """
    A simple 2-layer neural network implementation
    """
    def __init__(self, input_size, hidden_size, output_size):
        """Initialize network with random weights and biases"""
        # Initialize weights with small random values
        self.W1 = np.random.randn(hidden_size, input_size) * 0.01
        self.b1 = np.zeros((hidden_size, 1))
        
        self.W2 = np.random.randn(output_size, hidden_size) * 0.01
        self.b2 = np.zeros((output_size, 1))
    
    def forward(self, X, activation1=sigmoid, activation2=sigmoid):
        """
        Forward propagation through the network
        
        Parameters:
        - X: Input data (n_features, n_samples)
        - activation1: Activation function for hidden layer
        - activation2: Activation function for output layer
        
        Returns:
        - A2: Output predictions
        - cache: Values needed for backpropagation (if implemented)
        """
        # First layer
        Z1 = np.dot(self.W1, X) + self.b1
        A1 = activation1(Z1)
        
        # Output layer
        Z2 = np.dot(self.W2, A1) + self.b2
        A2 = activation2(Z2)
        
        # Store values for backpropagation
        cache = {
            "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2
        }
        
        return A2, cache
    
    def get_parameters(self):
        """Return network parameters"""
        return {
            "W1": self.W1,
            "b1": self.b1,
            "W2": self.W2,
            "b2": self.b2
        }

# Create a simple neural network with 2 inputs, 3 hidden neurons, and 1 output
nn = SimpleNeuralNetwork(input_size=2, hidden_size=3, output_size=1)

# Print network parameters
params = nn.get_parameters()
print("Network parameters:")
print(f"W1 shape: {params['W1'].shape}")
print(f"b1 shape: {params['b1'].shape}")
print(f"W2 shape: {params['W2'].shape}")
print(f"b2 shape: {params['b2'].shape}")

# Test with sample inputs
X = np.array([[0.5, 0.1, 0.8], 
              [0.2, 0.9, 0.4]])  # 2 features, 3 samples
predictions, cache = nn.forward(X)

print("\nForward propagation results:")
print(f"Input shape: {X.shape}")
print(f"Hidden layer activation shape: {cache['A1'].shape}")
print(f"Output shape: {predictions.shape}")
print(f"Predictions: {predictions.flatten()}")

# Show intermediate activations for the first sample
print("\nIntermediate values for first sample:")
print(f"Z1: {cache['Z1'][:,0]}")
print(f"A1: {cache['A1'][:,0]}")
print(f"Z2: {cache['Z2'][:,0]}")
print(f"A2 (output): {cache['A2'][:,0]}")

## 7. Visualizing Activation Functions

Let's create some interactive visualizations to gain a better understanding of how activation functions transform data and affect neural network behavior:

In [ ]:
# Interactive visualization of how activation functions transform data
def visualize_activation_transformation():
    """Interactive visualization of activation function transformations"""
    # Generate some random data
    np.random.seed(42)
    X = np.random.randn(100, 2) * 3
    
    # Create a meshgrid for visualization
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    
    # Function for interactive plot
    def plot_activation_boundary(w1=1.0, w2=1.0, bias=0.0, activation='sigmoid'):
        # Define activation functions dictionary
        act_functions = {
            'sigmoid': sigmoid,
            'tanh': tanh_activation,
            'relu': relu,
            'leaky_relu': leaky_relu
        }
        
        # Apply linear transformation and activation
        Z = np.zeros_like(xx)
        for i in range(xx.shape[0]):
            for j in range(xx.shape[1]):
                Z[i, j] = act_functions[activation](w1 * xx[i, j] + w2 * yy[i, j] + bias)
        
        # Linear boundary (before activation)
        linear_boundary = -bias/w2 - (w1/w2) * xx.ravel()
        valid_idx = (linear_boundary >= y_min) & (linear_boundary <= y_max)
        
        # Create plots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot original data and linear boundary
        ax1.scatter(X[:, 0], X[:, 1], c='blue', alpha=0.6)
        if np.any(valid_idx):
            ax1.plot(xx.ravel()[valid_idx], linear_boundary[valid_idx], 
                    'k-', label='Linear boundary')
        ax1.set_xlim(x_min, x_max)
        ax1.set_ylim(y_min, y_max)
        ax1.set_title('Input Space with Linear Boundary')
        ax1.legend()
        ax1.grid(True)
        
        # Plot transformed space
        contour = ax2.contourf(xx, yy, Z, 20, cmap='viridis', alpha=0.8)
        ax2.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', alpha=0.6)
        ax2.set_xlim(x_min, x_max)
        ax2.set_ylim(y_min, y_max)
        ax2.set_title(f'After {activation.capitalize()} Activation')
        plt.colorbar(contour, ax=ax2)
        
        # Display the equation
        equation = f"z = {w1:.2f}x₁ + {w2:.2f}x₂ + {bias:.2f}\noutput = {activation}(z)"
        plt.figtext(0.5, 0.01, equation, ha='center', fontsize=14, 
                   bbox={'facecolor': 'lightgray', 'alpha': 0.5, 'pad': 5})
        
        plt.tight_layout()
        plt.show()
    
    # Create interactive controls
    interact(plot_activation_boundary, 
             w1=widgets.FloatSlider(min=-3.0, max=3.0, step=0.1, value=1.0, description='Weight 1:'),
             w2=widgets.FloatSlider(min=-3.0, max=3.0, step=0.1, value=1.0, description='Weight 2:'),
             bias=widgets.FloatSlider(min=-5.0, max=5.0, step=0.5, value=0.0, description='Bias:'),
             activation=widgets.Dropdown(
                 options=['sigmoid', 'tanh', 'relu', 'leaky_relu'], 
                 value='sigmoid', 
                 description='Activation:'))

# Run the interactive visualization
visualize_activation_transformation()

### Impact of Activation Functions on Gradient Flow

One critical aspect of activation functions is how they affect gradient flow during backpropagation. Let's visualize this phenomenon:

In [ ]:
def visualize_gradient_flow():
    """Visualize how activation functions affect gradient flow in deep networks"""
    # Define a range of inputs
    x = np.linspace(-5, 5, 1000)
    
    # Create a figure
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()
    
    # Simulate gradient flow through multiple layers
    layer_counts = [1, 2, 5, 10]
    activation_funcs = [
        ('Sigmoid', sigmoid, sigmoid_derivative),
        ('Tanh', tanh_activation, tanh_derivative),
        ('ReLU', relu, relu_derivative),
        ('Leaky ReLU', leaky_relu, lambda x: leaky_relu_derivative(x))
    ]
    
    # For each activation function
    for i, (name, act_func, deriv_func) in enumerate(activation_funcs):
        ax = axs[i]
        
        # Plot for different network depths
        for num_layers in layer_counts:
            # Initialize with identity gradient
            gradient = np.ones_like(x)
            
            # Compute cascading effect of derivatives
            for _ in range(num_layers):
                gradient *= deriv_func(x)
            
            # Plot the resulting gradient
            ax.plot(x, gradient, label=f'{num_layers} layers', linewidth=2)
        
        ax.set_title(f'{name} Activation: Gradient Flow')
        ax.set_xlabel('Input Value')
        ax.set_ylabel('Gradient Magnitude')
        ax.grid(True)
        ax.legend()
        
        # Add horizontal line at y=0
        ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
        
        # Show vanishing gradient region for sigmoid/tanh
        if name in ['Sigmoid', 'Tanh']:
            ax.axhspan(-0.1, 0.1, alpha=0.2, color='red', label='Vanishing gradient region')
    
    plt.tight_layout()
    plt.show()

# Visualize gradient flow
visualize_gradient_flow()

## 8. Building a Simple Neural Network from Scratch

Now let's implement a simple neural network from scratch to demonstrate how neurons and activation functions work together. Our network will be trained to solve a simple classification problem.

In [ ]:
class NeuralNetwork:
    """Simple neural network with one hidden layer implementation from scratch"""
    
    def __init__(self, input_size, hidden_size, output_size):
        """Initialize the neural network with random weights"""
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Initialize weights and biases
        # Xavier/Glorot initialization for better convergence
        self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2. / input_size)
        self.b1 = np.zeros((hidden_size, 1))
        
        self.W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2. / hidden_size)
        self.b2 = np.zeros((output_size, 1))
        
        # Store parameters in a dictionary for easy access
        self.parameters = {
            "W1": self.W1, "b1": self.b1,
            "W2": self.W2, "b2": self.b2
        }
    
    def forward(self, X):
        """Forward propagation step"""
        # Retrieve parameters
        W1, b1 = self.parameters["W1"], self.parameters["b1"]
        W2, b2 = self.parameters["W2"], self.parameters["b2"]
        
        # First layer: ReLU activation
        Z1 = np.dot(W1, X) + b1
        A1 = relu(Z1)
        
        # Output layer: Sigmoid activation for binary classification
        Z2 = np.dot(W2, A1) + b2
        A2 = sigmoid(Z2)
        
        # Store values for backpropagation
        self.cache = {
            "X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2
        }
        
        return A2
    
    def compute_cost(self, Y, A2):
        """Compute the binary cross-entropy cost"""
        m = Y.shape[1]  # Number of examples
        
        # Binary cross-entropy loss
        cost = -1/m * np.sum(Y * np.log(A2) + (1 - Y) * np.log(1 - A2))
        
        return np.squeeze(cost)
    
    def backward(self, Y):
        """Backward propagation to compute gradients"""
        m = Y.shape[1]  # Number of examples
        
        # Retrieve cached values from forward propagation
        A2 = self.cache["A2"]
        A1 = self.cache["A1"]
        X = self.cache["X"]
        
        # Output layer gradient
        dZ2 = A2 - Y
        dW2 = 1/m * np.dot(dZ2, A1.T)
        db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
        
        # Hidden layer gradient
        dZ1 = np.dot(self.parameters["W2"].T, dZ2) * relu_derivative(self.cache["Z1"])
        dW1 = 1/m * np.dot(dZ1, X.T)
        db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)
        
        # Store gradients
        self.gradients = {
            "dW1": dW1, "db1": db1,
            "dW2": dW2, "db2": db2
        }
    
    def update_parameters(self, learning_rate):
        """Update parameters using gradient descent"""
        # Retrieve parameters and gradients
        W1, b1 = self.parameters["W1"], self.parameters["b1"]
        W2, b2 = self.parameters["W2"], self.parameters["b2"]
        
        dW1, db1 = self.gradients["dW1"], self.gradients["db1"]
        dW2, db2 = self.gradients["dW2"], self.gradients["db2"]
        
        # Update parameters
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        
        # Store updated parameters
        self.parameters = {
            "W1": W1, "b1": b1,
            "W2": W2, "b2": b2
        }
    
    def train(self, X, Y, num_iterations, learning_rate=0.1, print_cost=False):
        """Train the neural network"""
        costs = []
        
        for i in range(num_iterations):
            # Forward propagation
            A2 = self.forward(X)
            
            # Compute cost
            cost = self.compute_cost(Y, A2)
            
            # Backward propagation
            self.backward(Y)
            
            # Update parameters
            self.update_parameters(learning_rate)
            
            # Print cost every 100 iterations
            if print_cost and i % 100 == 0:
                print(f"Cost after iteration {i}: {cost}")
            
            if i % 100 == 0:
                costs.append(cost)
        
        return costs
    
    def predict(self, X):
        """Make predictions using trained model"""
        A2 = self.forward(X)
        predictions = np.round(A2)
        return predictions
    
    def accuracy(self, X, Y):
        """Calculate accuracy of predictions"""
        predictions = self.predict(X)
        return np.mean(predictions == Y)

Now let's generate some synthetic data to test our neural network implementation:

In [ ]:
# Generate synthetic data for classification
def generate_moons_data(n_samples=1000, noise=0.1):
    """Generate a synthetic two-class classification dataset"""
    from sklearn.datasets import make_moons
    
    # Generate data
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=42)
    
    # Reshape data for neural network
    X = X.T  # Shape: (2, n_samples)
    Y = y.reshape(1, n_samples)  # Shape: (1, n_samples)
    
    return X, Y

# Generate data
try:
    from sklearn.datasets import make_moons
    X, Y = generate_moons_data(n_samples=300, noise=0.2)
    sklearn_available = True
except ImportError:
    print("scikit-learn is not available. Generating simple data instead.")
    # Fallback to simple data if scikit-learn is not available
    np.random.seed(42)
    X = np.random.randn(2, 300)
    Y = np.array((X[0]**2 + X[1]**2 < 1.5)).reshape(1, 300)
    sklearn_available = False

# Plot the data
plt.figure(figsize=(10, 6))
plt.scatter(X[0, :], X[1, :], c=Y.ravel(), cmap=plt.cm.Spectral, s=40, edgecolors='k')
plt.title('Synthetic Classification Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar()
plt.grid(True)
plt.show()

In [ ]:
# Train our neural network on the synthetic data
def train_and_visualize():
    # Create a neural network with 2 inputs, 5 hidden neurons, and 1 output
    nn = NeuralNetwork(input_size=2, hidden_size=5, output_size=1)
    
    # Train the network
    costs = nn.train(X, Y, num_iterations=2000, learning_rate=0.5, print_cost=True)
    
    # Plot the cost over iterations
    plt.figure(figsize=(10, 4))
    plt.plot(costs)
    plt.title('Cost during Training')
    plt.xlabel('Iterations (hundreds)')
    plt.ylabel('Cost')
    plt.grid(True)
    plt.show()
    
    # Calculate accuracy
    accuracy = nn.accuracy(X, Y) * 100
    print(f"Accuracy: {accuracy:.2f}%")
    
    # Visualize decision boundary
    plt.figure(figsize=(10, 8))
    
    # Set min and max values and give it some padding
    x_min, x_max = X[0, :].min() - 1, X[0, :].max() + 1
    y_min, y_max = X[1, :].min() - 1, X[1, :].max() + 1
    
    # Generate a grid of points with distance h between them
    h = 0.01
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predict the function value for the whole grid
    Z = nn.predict(np.c_[xx.ravel(), yy.ravel()].T)
    Z = Z.reshape(xx.shape)
    
    # Plot the contour and training examples
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    plt.scatter(X[0, :], X[1, :], c=Y.ravel(), cmap=plt.cm.Spectral, s=40, edgecolors='k')
    plt.title('Neural Network Decision Boundary')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.grid(True)
    plt.colorbar()
    plt.show()
    
    # Return the trained model
    return nn

# Train and visualize the neural network
trained_nn = train_and_visualize()

### Visualizing Hidden Layer Activations

Let's visualize what each neuron in the hidden layer has learned:

In [ ]:
def visualize_hidden_neurons(nn):
    """Visualize what each neuron in the hidden layer has learned"""
    # Extract weights and biases of the first layer
    W1 = nn.parameters["W1"]  # Shape: (hidden_size, 2)
    b1 = nn.parameters["b1"]  # Shape: (hidden_size, 1)
    
    # Set min and max values for the grid
    x_min, x_max = X[0, :].min() - 1, X[0, :].max() + 1
    y_min, y_max = X[1, :].min() - 1, X[1, :].max() + 1
    
    # Generate a grid of points
    h = 0.01
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Create figure for visualization
    fig, axs = plt.subplots(1, nn.hidden_size, figsize=(15, 3))
    
    # For each neuron in hidden layer
    for i in range(nn.hidden_size):
        # Extract weights for this neuron
        w = W1[i, :].reshape(2, 1)
        b = b1[i, 0]
        
        # Calculate activation across the grid
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        Z = np.dot(grid_points, w) + b
        Z = relu(Z).reshape(xx.shape)
        
        # Plot the activation
        axs[i].contourf(xx, yy, Z, cmap='viridis', alpha=0.8)
        axs[i].scatter(X[0, :], X[1, :], c=Y.ravel(), cmap=plt.cm.Spectral, s=10, edgecolors='k', alpha=0.6)
        
        # Add a line showing the decision boundary for this neuron
        if w[1, 0] != 0:  # Avoid division by zero
            slope = -w[0, 0] / w[1, 0]
            intercept = -b / w[1, 0]
            x_boundary = np.array([x_min, x_max])
            y_boundary = slope * x_boundary + intercept
            axs[i].plot(x_boundary, y_boundary, 'k--', linewidth=1)
        
        axs[i].set_title(f'Neuron {i+1}')
        axs[i].set_xlim(x_min, x_max)
        axs[i].set_ylim(y_min, y_max)
        axs[i].grid(True)
    
    plt.tight_layout()
    plt.suptitle('Hidden Layer Neuron Activations', y=1.05, fontsize=16)
    plt.show()

# Visualize hidden layer activations
if 'trained_nn' in locals():
    visualize_hidden_neurons(trained_nn)

## Conclusion

In this notebook, we've explored the fundamental concepts of neural networks, focusing on neurons and activation functions. Here's a summary of what we've learned:

### Key Concepts Covered:

1. **Neural Network Architecture**:
   - Structure of layers: input, hidden, and output
   - How neurons connect to form networks

2. **Neurons**:
   - The computational units of neural networks
   - How they process inputs through weights, bias, and activation
   - Mathematical representation: $y = \sigma(\mathbf{w}^T\mathbf{x} + b)$

3. **Activation Functions**:
   - Purpose: introducing non-linearity to learn complex patterns
   - Common types: Sigmoid, Tanh, ReLU, Leaky ReLU, Softmax
   - Properties and use cases for each
   - Impact on gradient flow and neural network training

4. **Forward Propagation**:
   - How data flows through the network
   - Mathematical representation layer by layer

5. **Neural Network Implementation**:
   - Building a neural network from scratch
   - Training process with forward and backward propagation
   - Visualizing decision boundaries and neuron activations

### Why These Concepts Matter:

- **Foundation for Deep Learning**: Understanding neurons and activation functions is essential for working with more complex architectures like CNNs, RNNs, and Transformers.
- **Insight into Neural Network Behavior**: Knowledge of activation functions helps explain phenomena like vanishing/exploding gradients.
- **Better Model Design**: Choosing appropriate activation functions impacts model performance, convergence speed, and generalization.

### Next Steps in Your Learning Journey:

1. **Backpropagation**: Study how neural networks learn through gradient descent and backpropagation
2. **Optimization Techniques**: Explore advanced optimizers beyond basic gradient descent
3. **Regularization Methods**: Learn techniques to prevent overfitting
4. **Advanced Architectures**: Move on to CNNs, RNNs, and Transformer models
5. **Deep Learning Frameworks**: Apply these concepts using frameworks like TensorFlow and PyTorch

Understanding these neural network fundamentals provides the necessary foundation for diving deeper into the exciting world of deep learning.